<div style="background:#0A0E14;border:1px solid #22D3EE;border-radius:12px;overflow:hidden;font-family:system-ui,-apple-system,Segoe UI,sans-serif;">
  <div style="background:#0F1A2B;border-bottom:2px solid #22D3EE;color:#22D3EE;padding:6px 16px;font-family:Menlo,Consolas,monospace;letter-spacing:2px;font-size:12px;">AMS 261 · LAB 18 · ROVER FIELD TEST</div>
  <div style="padding:14px 16px;">
    <div style="font-size:24px;font-weight:700;color:#F8FAFC;">MISSION 18 // ROVER FIELD TEST</div>
    <div style="font-size:15px;color:#94A3B8;margin-top:4px;">Directional Derivatives and the Gradient</div>
  </div>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tancao1128/ams261-labs/blob/main/labs/AMS261-Lecture18.ipynb)
[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/tancao1128/ams261-labs/main?labpath=labs%2FAMS261-Lecture18.ipynb)
· [Download this notebook (.ipynb)](https://raw.githubusercontent.com/tancao1128/ams261-labs/main/labs/AMS261-Lecture18.ipynb)

## Mission briefing

A heat-seeking rover sits on a metal plate at $P=(2,-3)$. Coordinates are in cm.
The temperature is $T(x,y)=20-4x^2-y^2$ in °C.
The rover's sensor reads only two rates: $-16$ °C/cm toward the east and $+6$ °C/cm toward the north.
In this lab we test what these two readings tell the rover.

- **Time:** about 30 minutes.
- **When:** after class, before Lecture 19.
- **Before you start:** read Examples 1–5 and 7 and the MICRO-LAB on the Lecture 18 page.
- **Section numbers** (§1, §2, …) match the codenames on the lecture page.

**Lab objectives.** By the end of this lab you can

1. check the limit definition of $D_{\mathbf u}f$ against the shortcut $\nabla f\cdot\mathbf u$;
2. read the maximum, the minimum and the zeros of $D_{\mathbf u}T$ on its cosine curve;
3. show that a zero-change heading is tangent to the isotherm through $P$.

**Two angles.** We use two different angles. Keep them apart.

- $\phi$ is the **heading** of $\mathbf u$. It is measured counterclockwise from the positive $x$-axis, so $\mathbf u=\langle\cos\phi,\sin\phi\rangle$.
- $\theta$ is the **angle between** $\mathbf u$ and $\nabla T$. It runs from $0^\circ$ to $180^\circ$.

**How to work.** Run the cells from top to bottom with Shift+Enter. A cell marked ✏️ asks you to type something first.

> **Not graded. Practice only.**

In [ ]:
# SETUP: run this cell first. It needs no local files.
import base64, json, math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import Markdown, display

try:
    from ipywidgets import interact, FloatSlider
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

x, y = sp.symbols('x y', real=True)

# Colour tokens. Each colour has one meaning in every plot of this lab.
INK = '#1E293B'        # text, curves, the point P
GRAD = '#2563EB'       # the gradient direction (steepest ascent)
DESCENT = '#DC2626'    # the descent direction -grad T
ZERO = '#059669'       # zero-change (tangent) headings
DIRECTION = '#D97706'  # the chosen direction u
CONTOUR = '#94A3B8'    # level curves (isotherms)

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})


def show_math(latex):
    # Display one formula.
    display(Markdown(f"$${latex}$$"))


def show_text(text):
    # Display formatted text.
    display(Markdown(text))


def vec(v):
    # LaTeX for a vector written with angle brackets.
    return r'\left\langle ' + ',\ '.join(sp.latex(c) for c in v) + r'\right\rangle'


def fmt(v, digits=2):
    # Format a number; never print -0.00.
    v = float(v)
    if abs(v) < 0.5 * 10 ** (-digits):
        v = 0.0
    return f"{v:.{digits}f}"


def draw_arrow(ax, start, vec, color, ls='-', lw=2.2):
    # Draw one arrow from start along vec.
    ax.annotate('', xy=(start[0] + vec[0], start[1] + vec[1]), xytext=(start[0], start[1]),
                arrowprops=dict(arrowstyle='-|>', color=color, lw=lw, ls=ls, mutation_scale=16))


def legend_line(color, label, ls='-', marker=None):
    # A legend entry for an arrow or a marker.
    return Line2D([0], [0], color=color, ls=ls, lw=2.2 if marker is None else 0,
                  marker=marker, markersize=8, label=label)


def as_number(v):
    # Accept a number or a string such as "2*sqrt(73)".
    return float(sp.sympify(v))


def check_answer(given, target, tol=5e-3):
    # Compare an answer with a target. Returns True or False; never raises.
    try:
        if given is None:
            return False
        if isinstance(target, bool):
            return given is target
        if isinstance(target, (list, tuple)):
            return (len(given) == len(target)
                    and all(check_answer(a, b, tol) for a, b in zip(given, target)))
        return math.isclose(as_number(given), float(target), rel_tol=1e-3, abs_tol=tol)
    except Exception:
        return False


print('Setup complete. Widgets available:', HAS_WIDGETS)

---
## PREDICT · §1 SENSOR SWEEP

The sensor gives two rates at $P$:

- toward the east (heading $\phi=0^\circ$): $T_x(P)=-16$ °C/cm;
- toward the north (heading $\phi=90^\circ$): $T_y(P)=+6$ °C/cm.

The rover can move in any direction. **Do not compute yet.** Make two guesses.

1. At which heading $\phi$ does the temperature rise fastest?
2. How many headings give zero change in temperature: 0, 1, 2, or infinitely many?

A rough guess is fine. You will check it in §6.

In [ ]:
# ✏️ PREDICT. Type your two guesses, then run this cell. Do not compute first.
my_phi_max_guess = None      # heading phi in degrees (0 to 360) of fastest warming
my_zero_count_guess = None   # number of zero-change headings: 0, 1, 2 or "infinitely many"

if my_phi_max_guess is None or my_zero_count_guess is None:
    print('Prediction not logged yet. Type two guesses above and run this cell again.')
else:
    print(f'Prediction logged: phi_max = {my_phi_max_guess} deg, zero-change headings: {my_zero_count_guess}.')
    print('You will check it in §6.')

In [ ]:
# The plate model. Units: T in °C, x and y in cm.
T_expr = 20 - 4*x**2 - y**2
P = (2, -3)
sensor_east = -16    # T_x(P) in °C/cm
sensor_north = 6     # T_y(P) in °C/cm

T_at_P = T_expr.subs({x: P[0], y: P[1]})
print(f'T(P) = {T_at_P} °C at P = {P}.')

---
## EXPLORE · §2 DEFINITION CORE

For a unit vector $\mathbf u=\langle a,b\rangle$, the directional derivative at $(x_0,y_0)$ is

$$
D_{\mathbf u}f(x_0,y_0)=\lim_{t\to 0}\frac{f(x_0+ta,\,y_0+tb)-f(x_0,y_0)}{t}.
$$

The first engine, `directional_derivative1`, evaluates this limit with sympy.
It refuses a direction that is not a unit vector. This guards against the most common error.

**Page Example 1.** Let $f(x,y)=e^x\cos y$, and let $\mathbf u$ be the unit vector at heading $\phi=\pi/4$.
(The page calls this angle $\theta$. It is measured from the $x$-axis, so in this lab it is $\phi$.)
Find $D_{\mathbf u}f(0,0)$.

In [ ]:
def directional_derivative1(f, x_0, y_0, a, b):
    # Directional derivative from the limit definition. (a, b) must be a unit vector.
    if abs(float(sp.N(a**2 + b**2 - 1))) > 1e-9:   # tolerance, so float unit vectors pass
        raise ValueError('Normalize the direction first: a^2 + b^2 must equal 1.')
    t = sp.symbols('t')
    D = (f(x_0 + t*a, y_0 + t*b) - f(x_0, y_0)) / t
    return sp.limit(D, t, 0)


f1 = lambda X, Y: sp.exp(X) * sp.cos(Y)
a1, b1 = sp.cos(sp.pi/4), sp.sin(sp.pi/4)

d1 = directional_derivative1(f1, 0, 0, a1, b1)
show_math(r'D_{\mathbf u}f(0,0)\ \text{(limit)} = ' + sp.latex(sp.simplify(d1)))

---
## EXPLORE · §3 SHORTCUT PROTOCOL

If $f$ is differentiable at $P$ and $\mathbf u$ is a unit vector, then

$$
D_{\mathbf u}f(P)=\nabla f(P)\cdot\mathbf u .
$$

The second engine, `directional_derivative2`, uses this shortcut. It normalizes the direction first.
The cell below runs it on the same Example 1 and **checks** that the two engines agree.

In [ ]:
def compute_gradient(f, variables, point=None):
    # Gradient of f; evaluated at point if a point is given.
    grad_f = [sp.diff(f, var) for var in variables]
    if point is not None:
        grad_f = [gi.subs(dict(zip(variables, point))) for gi in grad_f]
    return grad_f


def directional_derivative2(f, variables, point, direction):
    # Directional derivative via the gradient and the dot product.
    grad_f = compute_gradient(f, variables, point)
    length = sp.sqrt(sum(d**2 for d in direction))
    u = [d / length for d in direction]   # normalize the direction
    return sum(gi * ui for gi, ui in zip(grad_f, u))


d2 = directional_derivative2(sp.exp(x) * sp.cos(y), [x, y], [0, 0], [a1, b1])
show_math(r'D_{\mathbf u}f(0,0)\ \text{(shortcut)} = ' + sp.latex(sp.simplify(d2)))
print('Engines agree on Example 1:', sp.simplify(d1 - d2) == 0)

**Now test the engines on the rover.** The cell below compares the two engines for $T$ at $P$ along four headings.
East and north should reproduce the two sensor readings.

In [ ]:
T_func = lambda X, Y: T_expr.subs({x: X, y: Y}, simultaneous=True)

headings = {
    'east, phi = 0 deg':    (sp.Integer(1), sp.Integer(0)),
    'north, phi = 90 deg':  (sp.Integer(0), sp.Integer(1)),
    'phi = 30 deg':         (sp.cos(sp.pi/6), sp.sin(sp.pi/6)),
    'phi = 135 deg':        (sp.cos(3*sp.pi/4), sp.sin(3*sp.pi/4)),
}

print(f"{'heading':<22}{'limit':>12}{'shortcut':>12}   match")
for name, (a, b) in headings.items():
    lim = sp.simplify(directional_derivative1(T_func, P[0], P[1], a, b))
    sc = sp.simplify(directional_derivative2(T_expr, [x, y], list(P), [a, b]))
    print(f"{name:<22}{fmt(lim, 4):>12}{fmt(sc, 4):>12}   {sp.simplify(lim - sc) == 0}")

---
## EXPLORE · §4 GRADIENT CORE

The gradient collects the two coordinate rates:

$$
\nabla T(P)=\langle T_x(P),\,T_y(P)\rangle .
$$

So the two sensor readings already **are** the gradient. The cell below builds $\nabla T(P)$ from the sensors.
Then it compares the result with sympy's gradient of $T$.

In [ ]:
gradT_sensors = sp.Matrix([sensor_east, sensor_north])
gradT_formula = sp.Matrix(compute_gradient(T_expr, [x, y]))
gradT_P = gradT_formula.subs({x: P[0], y: P[1]})
gradT_norm = sp.sqrt(gradT_P.dot(gradT_P))

show_math(r'\nabla T(x,y) = ' + vec(gradT_formula))
show_math(r'\nabla T(P)\ \text{from the sensors} = ' + vec(gradT_sensors)
          + r',\qquad \nabla T(P)\ \text{from sympy} = ' + vec(gradT_P))
print('Sensors agree with sympy:', gradT_sensors == gradT_P)
show_math(r'\|\nabla T(P)\| = ' + sp.latex(sp.simplify(gradT_norm)) + r'\approx ' + fmt(gradT_norm)
          + r'\ \ {}^\circ\mathrm{C/cm}')

# Numerical copies for the plots below.
g = np.array([float(gradT_P[0]), float(gradT_P[1])])
g_norm = float(np.hypot(g[0], g[1]))
phi0 = math.degrees(math.atan2(g[1], g[0])) % 360   # heading of grad T

---
## EXPLAIN · §6 STEEPEST-VECTOR PROTOCOL

Write $\mathbf u=\langle\cos\phi,\sin\phi\rangle$. Then

$$
D_{\mathbf u}T(P)=T_x(P)\cos\phi+T_y(P)\sin\phi=\|\nabla T(P)\|\cos\theta ,
$$

where $\theta$ is the angle between $\mathbf u$ and $\nabla T(P)$.
As $\phi$ makes one full turn, the graph of $D_{\mathbf u}T(P)$ is one cosine wave.

**Your task.** Move the slider $\phi$ through a full turn. Find

- the heading where the curve peaks, and the peak value;
- the heading where the curve is lowest, and that value;
- the two headings where the curve crosses zero.

If the slider does not appear, use the static cell after it. Change `phi_static` there and run it again.

In [ ]:
def plot_steepest(phi_deg=0.0):
    p = math.radians(phi_deg)
    u = np.array([math.cos(p), math.sin(p)])
    D = float(g @ u)
    theta = math.degrees(math.acos(float(np.clip(g @ u / g_norm, -1.0, 1.0))))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.8),
                                   gridspec_kw={'width_ratios': [1.25, 1]})

    # Left panel: the cosine curve of D_u T(P) against the heading phi.
    phis = np.linspace(0, 360, 721)
    ax1.plot(phis, g[0]*np.cos(np.radians(phis)) + g[1]*np.sin(np.radians(phis)),
             color=INK, lw=2, label=r'$D_{\mathbf{u}}T(P)$ for every heading $\phi$')
    ax1.axhline(0, color=CONTOUR, lw=1)
    ax1.axvline(phi_deg, color=DIRECTION, ls=':', lw=1.2)
    ax1.plot([phi_deg], [D], 'o', color=DIRECTION, ms=10, label='current heading of u (amber dot)')
    ax1.set_xlim(0, 360)
    ax1.set_xticks(range(0, 361, 45))
    ax1.set_xlabel(r'heading $\phi$ of $\mathbf{u}$ (degrees from the $+x$-axis)')
    ax1.set_ylabel(r'$D_{\mathbf{u}}T(P)$  (°C/cm)')
    ax1.set_title(rf'$\phi$ = {phi_deg:.0f}°,  $\theta$ = {theta:.1f}°,  '
                  rf'$D_{{\mathbf{{u}}}}T(P)$ = {fmt(D)} °C/cm')
    ax1.grid(alpha=0.3)
    ax1.legend(loc='lower left', fontsize=9)

    # Right panel: a compass at P.
    s = np.linspace(0, 2*np.pi, 200)
    ax2.plot(np.cos(s), np.sin(s), color=CONTOUR, lw=1)
    gu = g / g_norm
    tz = np.array([-gu[1], gu[0]])            # a unit vector perpendicular to grad T
    draw_arrow(ax2, (0, 0), gu, GRAD)
    draw_arrow(ax2, (0, 0), -gu, DESCENT, ls='-.')
    draw_arrow(ax2, (0, 0), tz, ZERO, ls='--')
    draw_arrow(ax2, (0, 0), -tz, ZERO, ls='--')
    draw_arrow(ax2, (0, 0), u, DIRECTION, lw=2.8)
    # Text labels at the arrow tips, so colour is never the only signal.
    ax2.text(*(1.18*gu), r'$\nabla T$', ha='center', va='center', fontsize=11)
    ax2.text(*(-1.18*gu), r'$-\nabla T$', ha='center', va='center', fontsize=11)
    ax2.text(*(1.12*u + np.array([0.0, 0.1])), r'$\mathbf{u}$', ha='center', va='center', fontsize=11)
    # The arc that shows theta, the angle between u and grad T.
    d = ((phi_deg - phi0 + 180) % 360) - 180
    arc = np.radians(phi0 + np.linspace(0, d, 60))
    ax2.plot(0.4*np.cos(arc), 0.4*np.sin(arc), color=INK, lw=1.4)
    ax2.plot([0], [0], 'o', color=INK, ms=6)
    ax2.set_xlim(-1.35, 1.35)
    ax2.set_ylim(-1.35, 1.35)
    ax2.set_aspect('equal')
    ax2.grid(alpha=0.3)
    ax2.set_xlabel('east component')
    ax2.set_ylabel('north component')
    ax2.set_title('Compass at P (unit vectors)')
    ax2.legend(handles=[
        legend_line(GRAD, r'blue: direction of $\nabla T(P)$ (fastest warming)'),
        legend_line(DESCENT, r'red dash-dot: direction of $-\nabla T(P)$ (fastest cooling)', ls='-.'),
        legend_line(ZERO, 'green dashed: zero-change headings', ls='--'),
        legend_line(DIRECTION, r'amber: your direction $\mathbf{u}$'),
        legend_line(INK, r'black arc: angle $\theta$ between $\mathbf{u}$ and $\nabla T$'),
    ], loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=9, frameon=False)
    fig.tight_layout()
    display(fig)      # display, then close: works the same with or without the slider
    plt.close(fig)


if HAS_WIDGETS:
    interact(plot_steepest,
             phi_deg=FloatSlider(min=0, max=360, step=1, value=0, description='φ (deg)',
                                 continuous_update=False))
else:
    print('ipywidgets is not available here. Use the static cell below instead.')

*Figure description.* Left: the curve of $D_{\mathbf u}T(P)$ against the heading $\phi$ is one cosine wave. An amber dot marks the current heading, and the title prints $\phi$, $\theta$ and the rate. Right: a compass at $P$. The blue arrow is the direction of $\nabla T(P)$, the red arrow is the opposite direction, the two green dashed arrows are the zero-change headings, and the amber arrow is $\mathbf u$. A black arc marks the angle $\theta$.

In [ ]:
# Static version (no widgets needed). Change phi_static and run again.
phi_static = 45
plot_steepest(phi_static)

*Figure description.* The same two panels as above, drawn once at the heading $\phi$ stored in `phi_static`.

<div style="background:#EFF6FF;border-left:6px solid #2563EB;padding:8px 16px;border-radius:8px;">

<strong>🧠 Explain.</strong> Read off the heading $\phi$ where the curve peaks and the value of $D_{\mathbf u}T$ there.
<br><br>Now explain <b>in one sentence</b>, using $D_{\mathbf u}T=\nabla T\cdot\mathbf u=\|\nabla T\|\cos\theta$: why is the maximum reached exactly when $\mathbf u$ points along $\nabla T$ (that is, $\theta=0$)? Why is the maximum value exactly $\|\nabla T(P)\|=2\sqrt{73}\approx 17.09$ °C/cm?
<br><br>Also read the two headings where the curve crosses zero. What is $\theta$ there? What does $D_{\mathbf u}T=0$ say about $\mathbf u$ and the level curve through $P$? You will test your answer in §7.

</div>

Write your sentence first. Then run the next cell to compare it with your §1 prediction.

In [ ]:
# Check your §1 prediction against the curve.
zero_headings = sorted([(phi0 - 90) % 360, (phi0 + 90) % 360])
print(f'Peak heading:      phi_max = {phi0:.2f} deg, where D_u T = {fmt(g_norm)} °C/cm (theta = 0).')
print(f'Lowest heading:    phi_min = {(phi0 + 180) % 360:.2f} deg, where D_u T = {fmt(-g_norm)} °C/cm (theta = 180 deg).')
print(f'Zero-change headings: {zero_headings[0]:.2f} deg and {zero_headings[1]:.2f} deg (theta = 90 deg).')
print()
if my_phi_max_guess is None:
    print('You did not log a prediction in §1. Next time, commit a guess first.')
else:
    try:
        miss = abs(((float(my_phi_max_guess) - phi0 + 180) % 360) - 180)
        print(f'Your guess for phi_max was off by {miss:.1f} deg.')
    except (TypeError, ValueError):
        print('Log your guess as a number of degrees, for example 120, then run this cell again.')
if my_zero_count_guess is not None:
    print('Your zero-count guess was', my_zero_count_guess, '- the curve crosses zero twice per turn.')

---
## CHALLENGE · §7 CONTOUR LOCK

**Theorem (gradient normal to a level curve).** If $f$ is differentiable at $P$ and $\nabla f(P)\neq\mathbf 0$, then $\nabla f(P)$ is normal to the regular level curve through $P$.

Since $T(P)=-5$, the isotherm through $P$ is $20-4x^2-y^2=-5$, that is,

$$
4x^2+y^2=25 .
$$

We compare two headings for the rover.

- **Heading A** points straight at the hot centre $(0,0)$: $\mathbf u_A=\langle-2,3\rangle/\sqrt{13}$.
- **Heading B** points along the gradient: $\mathbf u_B=\nabla T(P)/\|\nabla T(P)\|$.

The cell below computes both rates. It also finds the two unit headings perpendicular to $\nabla T(P)$.
Then it checks that they are tangent to the isotherm.

In [ ]:
uA = sp.Matrix([-2, 3]) / sp.sqrt(13)
uB = gradT_P / gradT_norm
DA = sp.nsimplify(gradT_P.dot(uA))
DB = sp.simplify(gradT_P.dot(uB))
show_math(r'D_{\mathbf u_A}T(P) = ' + sp.latex(DA) + r'\approx ' + fmt(DA)
          + r',\qquad D_{\mathbf u_B}T(P) = ' + sp.latex(DB) + r'\approx ' + fmt(DB))

# Zero-change headings: rotate grad T by +90 and -90 degrees, then normalize.
t1 = sp.Matrix([-gradT_P[1], gradT_P[0]]) / gradT_norm
t1 = -t1 if t1[0] < 0 else t1                      # list the heading with a positive x-part first
t2 = -t1
for name, tv in [('t1', t1), ('t2', t2)]:
    heading = math.degrees(math.atan2(float(tv[1]), float(tv[0]))) % 360
    show_math(r'\mathbf ' + name[0] + '_' + name[1] + ' = ' + r'\frac{1}{\sqrt{73}}' + vec(sp.simplify(tv * sp.sqrt(73)))
              + r',\quad \phi\approx ' + f'{heading:.2f}^\\circ'
              + r',\quad \nabla T(P)\cdot\mathbf ' + name[0] + '_' + name[1] + ' = '
              + sp.latex(sp.simplify(gradT_P.dot(tv))))

# Tangency check: the slope of the isotherm at P, by implicit differentiation.
slope = sp.idiff(4*x**2 + y**2 - 25, y, x).subs({x: P[0], y: P[1]})
print('Slope of the isotherm at P:', slope, '| slope of t1:', sp.simplify(t1[1] / t1[0]))
print('Zero-change headings are tangent to the isotherm:', sp.simplify(slope - t1[1] / t1[0]) == 0)

In [ ]:
def contour_map(ax):
    # Isotherms of T, the isotherm through P, the hot centre and P.
    X, Y = np.meshgrid(np.linspace(-3.6, 3.6, 300), np.linspace(-5.6, 5.6, 300))
    Z = 20 - 4*X**2 - Y**2
    cs = ax.contour(X, Y, Z, levels=np.arange(-30, 20, 10), colors=CONTOUR,
                    linewidths=1, linestyles='solid')
    ax.clabel(cs, fmt='%d °C', fontsize=8)
    s = np.linspace(0, 2*np.pi, 400)
    ax.plot(2.5*np.cos(s), 5*np.sin(s), color=INK, lw=2.4)
    ax.plot([0], [0], marker='*', color=INK, ms=14, ls='none')
    ax.plot([P[0]], [P[1]], 'o', color=INK, ms=8)
    ax.annotate('P', (P[0], P[1]), xytext=(8, -12), textcoords='offset points', color=INK)
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-5.6, 5.6)
    ax.set_aspect('equal')
    ax.set_xlabel('x (cm)')
    ax.set_ylabel('y (cm)')
    ax.grid(alpha=0.2)


base_legend = [
    legend_line(CONTOUR, 'grey: isotherms, labelled in °C'),
    legend_line(INK, r'thick black: isotherm $4x^2+y^2=25$ through P (T = -5 °C)'),
    legend_line(INK, 'black star: hot centre (0, 0), T = 20 °C', marker='*'),
    legend_line(INK, 'black dot: rover at P = (2, -3)', marker='o'),
]

fig, ax = plt.subplots(figsize=(10.5, 7))
contour_map(ax)
L = 1.6
draw_arrow(ax, P, L * np.array([float(uB[0]), float(uB[1])]), GRAD)
draw_arrow(ax, P, L * np.array([float(uA[0]), float(uA[1])]), DIRECTION)
draw_arrow(ax, P, L * np.array([float(t1[0]), float(t1[1])]), ZERO, ls='--')
draw_arrow(ax, P, L * np.array([float(t2[0]), float(t2[1])]), ZERO, ls='--')
ax.set_title('§7 Contour lock: headings at P on the isotherm map')
ax.legend(handles=base_legend + [
    legend_line(GRAD, r'blue: heading B, along $\nabla T(P)$, ' + f'rate {fmt(DB)} °C/cm'),
    legend_line(DIRECTION, r'amber: heading A, toward the hot centre, ' + f'rate {fmt(DA)} °C/cm'),
    legend_line(ZERO, 'green dashed: zero-change headings (tangent to the isotherm)', ls='--'),
], loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=9, frameon=False)
plt.tight_layout()
plt.show()

*Figure description.* A contour map of $T$ with equal scales on both axes. Grey ellipses are isotherms, and the thick black ellipse is the isotherm $4x^2+y^2=25$ through $P$. At $P$, the blue arrow (heading B) points along $\nabla T(P)$ and crosses the isotherm at a right angle. The amber arrow (heading A) points at the hot centre, marked by a star. The two green dashed arrows lie along the isotherm.

<div style="background:#FFFBEB;border-left:6px solid #D97706;padding:8px 16px;border-radius:8px;">

<strong>🎯 Challenge question.</strong> Heading A points straight at the hottest spot. Yet its rate is about $13.87$ °C/cm, while heading B gives about $17.09$ °C/cm. Why is the straight-line heading <b>not</b> the fastest way to warm up <i>right now</i>? Answer in two or three sentences. Use the shape of the isotherms in your answer.

</div>

<details>
<summary>Hint (open after you try)</summary>

The isotherms are ellipses, not circles. They are packed more tightly in the $x$-direction, since $T$ has $-4x^2$ but only $-y^2$. Which heading crosses the isotherms at a right angle? What is the angle $\theta$ between $\mathbf u_A$ and $\nabla T(P)$?

</details>

### Rover trace: follow $\nabla T$ for a few steps

Now let the rover keep choosing heading B. At each step it reads $\nabla T$ at its current position.
Then it moves a short distance $h$ along the unit vector $\nabla T/\|\nabla T\|$.
This is Euler's method with small steps.

**Direction, not speed.** We normalize $\nabla T$ at every step. So the trace shows the *path* of steepest ascent only. The step length $h$ is our choice; it is not a time or a speed.

In [ ]:
def grad_T(px, py):
    return np.array([-8.0 * px, -2.0 * py])


h = 0.1          # step length in cm
n_steps = 25     # 25 steps of 0.1 cm
path = [np.array(P, dtype=float)]
for k in range(n_steps):
    gv = grad_T(*path[-1])
    if np.hypot(*gv) < 1e-12:      # the gradient vanishes: no steepest direction
        break
    path.append(path[-1] + h * gv / np.hypot(*gv))
path = np.array(path)
T_path = 20 - 4*path[:, 0]**2 - path[:, 1]**2

print(f"{'step':>4}{'x':>9}{'y':>9}{'T (°C)':>10}")
for k in list(range(0, 5)) + [len(path) - 1]:
    print(f"{k:>4}{path[k, 0]:>9.3f}{path[k, 1]:>9.3f}{T_path[k]:>10.2f}")
print('T rises at every step:', bool(np.all(np.diff(T_path) > 0)))

fig, ax = plt.subplots(figsize=(10.5, 7))
contour_map(ax)
ax.plot(path[:, 0], path[:, 1], '-o', color=GRAD, ms=3, lw=1.8)
ax.set_title(f'Rover trace: {len(path) - 1} Euler steps of h = {h} cm along the unit gradient')
ax.legend(handles=base_legend + [
    legend_line(GRAD, r'blue dots: rover positions, each step along $\nabla T/\|\nabla T\|$', marker='o'),
], loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=9, frameon=False)
plt.tight_layout()
plt.show()

*Figure description.* The same contour map. A chain of blue dots starts at $P$ and moves inward toward the hot centre. The chain bends: each step leaves its isotherm at a right angle, so the path is not the straight segment from $P$ to $(0,0)$.

*Note.* This lab only draws a numerical trace that starts at $P$. The exact path needs a different method: see Example 6 on the lecture page.

---
## TRANSFER

<div style="background:#EFF6FF;border-left:6px solid #2563EB;padding:8px 16px;border-radius:8px;">

<strong>🚀 Transfer, without the slider.</strong> Page Example 7 uses $f(x,y)=y-\sin x$. Its level curve $f=0$ is $y=\sin x$, and $\nabla f(\pi,0)=\langle 1,1\rangle$.

1. Use only $D_{\mathbf u}f=\|\nabla f\|\cos\theta$. Find <b>both</b> headings $\phi$ at which $D_{\mathbf u}f(\pi,0)=0$.
2. Level-curve normal check. The tangent vector of $y=\sin x$ at $x=\pi$ is $\langle 1,\cos\pi\rangle$. Show that $\nabla f(\pi,0)$ is perpendicular to it. Is one of your zero headings parallel to this tangent?

</div>

In [ ]:
# ✏️ Optional check for part 2. Fill in both vectors, then run.
grad_f_at_pi = None    # for example sp.Matrix([?, ?])
tangent_at_pi = None   # tangent vector of y = sin x at x = pi

if grad_f_at_pi is None or tangent_at_pi is None:
    print('Fill in both vectors first.')
else:
    print('dot product =', sp.simplify(grad_f_at_pi.dot(tangent_at_pi)))

<details>
<summary>Check your answer</summary>

$\nabla f(\pi,0)=\langle1,1\rangle$ has heading $\phi_0=45^\circ$. The zero headings are $\phi_0\pm 90^\circ$, that is, $135^\circ$ and $315^\circ$. Indeed, $D_{\mathbf u}f=\cos\phi+\sin\phi=0$ at both headings.
The tangent vector is $\langle 1,\cos\pi\rangle=\langle1,-1\rangle$, and $\langle1,1\rangle\cdot\langle1,-1\rangle=0$. Its heading is $315^\circ$. So the zero-change headings are exactly the two tangent directions of the level curve, as in §7.

</details>

---
## CREATE · Mission log

Fill in the log with your own readings from §6 and §7. Use degrees for headings and °C/cm for rates.
Then write one sentence in your own words.

In [ ]:
# ✏️ CREATE. Fill in your mission log, then run this cell and the next one.
mission_log = {
    'phi_max': None,                 # heading phi (deg) of fastest warming at P
    'D_max': None,                   # the largest D_u T(P), in °C/cm
    'zero_headings': [None, None],   # the two headings phi (deg) with D_u T(P) = 0
    'one_sentence': '',              # why are the zero headings tangent to the isotherm through P?
}

In [ ]:
# Check the mission log. It prints only "correct" or "try again".
def same_heading(a, b, tol=0.5):
    try:
        return abs(((as_number(a) - b + 180) % 360) - 180) < tol
    except Exception:
        return False

ok_phi = same_heading(mission_log['phi_max'], phi0)
ok_D = check_answer(mission_log['D_max'], g_norm, tol=0.01)
zh = mission_log['zero_headings']
ok_zero = (len(zh) == 2
           and ((same_heading(zh[0], zero_headings[0]) and same_heading(zh[1], zero_headings[1]))
                or (same_heading(zh[0], zero_headings[1]) and same_heading(zh[1], zero_headings[0]))))
for key, ok in [('phi_max', ok_phi), ('D_max', ok_D), ('zero_headings', ok_zero)]:
    print(f'{key}:', 'correct' if ok else 'try again')
sentence = mission_log['one_sentence'].strip()
print('one_sentence:', 'logged' if sentence else 'write one sentence')

<div style="background:#0A0E14;border:1px solid #22D3EE;border-radius:10px;padding:10px 16px;color:#E2E8F0;">
<span style="color:#22D3EE;font-family:Menlo,Consolas,monospace;letter-spacing:1px;font-size:12px;">PREDICTION FOR LECTURE 19</span><br>
If the gradient is perpendicular to every level curve, what should it be for a level <b>surface</b> $F(x,y,z)=c$?
<br>Write your guess in one sentence. Lecture 19 will test it.
</div>

---
## ⚠️ Common Errors: check yourself against these

<div style="background:#FEF2F2;border-left:6px solid #DC2626;padding:8px 16px;border-radius:8px;">

- **Forgetting to normalize.** $D_{\mathbf u}f=\nabla f\cdot\mathbf u$ holds only for a **unit** $\mathbf u$. If you skip the normalization, your answer is scaled by $\|\mathbf v\|$. This is why the §2 engine refuses a non-unit direction.
- **Giving the direction of maximum increase as a non-unit vector.** "The direction is $\langle-16,6\rangle$" is incomplete when a unit direction is asked for. The unit direction is $\langle-8,3\rangle/\sqrt{73}$.
- **Heading straight at the maximum.** The fastest local heading is along $\nabla T$. It is not the straight line to the hottest point (§7).
- **Drawing $\nabla f$ tangent to the level curve.** It is **normal** to it. In §7, $D_{\mathbf u}T=0$ exactly at the two headings tangent to the isotherm. These headings are $90^\circ$ away from $\nabla T$.
- **Mixing up $\phi$ and $\theta$.** $\phi$ is measured from the $x$-axis. $\theta$ is measured from $\nabla f$. The rate is $\|\nabla f\|\cos\theta$, not $\|\nabla f\|\cos\phi$.
- **Assuming that a directional derivative exists because the partial derivatives do.** This is not guaranteed without differentiability. The standing counterexample is $f(x,y)=\sqrt[3]{xy}$ (see the lecture page).

</div>

## 📝 Retrieval check: mini-quiz

Fill in `student_answers` from memory, then run the cell. Numbers may be decimals (three digits are enough) or exact strings such as `"2*sqrt(73)"`.
The check prints only **correct** or **try again**. After your second try you may set `reveal = True` to see the key.

In [ ]:
# ✏️ Mini-quiz for T(x, y) = 20 - 4x^2 - y^2 at P = (2, -3).
student_answers = {
    'grad_T_at_P': None,                        # [T_x(P), T_y(P)]
    'max_rate_T': None,                         # largest D_u T(P), in °C/cm
    'D_u_T_dir_3_4': None,                      # D_u T(P) in the direction of the vector <3, 4>
    'steepest_descent_unit': None,              # unit direction of fastest cooling at P, as [u1, u2]
    'gradient_is_normal_to_level_curve': None,  # True or False
}
reveal = False   # set to True only after your second try

_key = json.loads(base64.b64decode('eyJncmFkX1RfYXRfUCI6IFstMTYsIDZdLCAibWF4X3JhdGVfVCI6IDE3LjA4ODAwNzQ5MDYzNTA2LCAiRF91X1RfZGlyXzNfNCI6IC00LjgsICJzdGVlcGVzdF9kZXNjZW50X3VuaXQiOiBbMC45MzYzMjkxNzc1NjkwNDQ1LCAtMC4zNTExMjM0NDE1ODgzOTE3XSwgImdyYWRpZW50X2lzX25vcm1hbF90b19sZXZlbF9jdXJ2ZSI6IHRydWV9'))
if any(v is not None for v in student_answers.values()):
    quiz_attempts = globals().get('quiz_attempts', 0) + 1
else:
    quiz_attempts = globals().get('quiz_attempts', 0)

score = 0
for name, target in _key.items():
    ok = check_answer(student_answers[name], target)
    score += ok
    print(f'{name}:', 'correct' if ok else 'try again')
print(f'Score: {score}/{len(_key)}   (attempt {quiz_attempts})')

if reveal and quiz_attempts < 2:
    print('Try at least twice before you reveal the key.')
elif reveal:
    print('Key:', _key)
elif score == len(_key):
    print('All correct. You are ready for Lecture 19.')

## 🤖 AI Learning Companion

<div style="background:#ECFDF5;border-left:6px solid #059669;padding:6px 14px;border-radius:8px;">
<strong>Student-facing AI contract — paste this at the start of every AI session</strong>
<br>1. Ask me what I have already tried before you explain anything.
<br>2. Do not give a complete solution until I have made a prediction or proposed a next step.
<br>3. Point to the first gap in my reasoning, not every later consequence.
<br>4. Make me verify the result by units, sign, substitution, a graph, or a limiting case.
<br>5. End with one transfer question that changes the context.
</div>

<div style="background:#F8FAFC;border-left:6px solid #94A3B8;padding:6px 14px;border-radius:8px;margin-top:6px;">
<strong>Verification rule.</strong> Treat an AI answer as a conjecture until it agrees with the definitions, theorems, and figures on this page; bring disagreements to class. Neither an AI response nor an illustration overrides a checked definition or proof.
</div>

**Prompt for this lab: Error Hunter.** Paste your own gradient or directional-derivative solution in place of `[paste my work]`.

```
Here is my solution to a directional-derivative or gradient problem:
[paste my work]
Find the FIRST invalid step. Do not fix it. Instead:
(1) quote that step, (2) name the underlying misconception in one sentence,
(3) give me a two-line counterexample or sanity check that exposes it,
(4) ask me to repair the step myself.
If my solution is correct, instead give me a harder variant that breaks the
method I used, and ask me why it breaks.
```

## 🔒 Independence Check

<div style="background:#ECFDF5;border-left:6px solid #059669;padding:8px 16px;border-radius:8px;">
<i>No AI, no hints, new numbers. This shows that you can do it alone.</i>
</div>

Let $g(x,y)=x^2y-3x+y^2$.

1. Find $\nabla g(1,2)$.
2. Find the unit direction of steepest **decrease** at $(1,2)$.
3. Find the maximum rate of increase at $(1,2)$.

Do this on paper, without AI and without the worked examples. It is not checked here. It is practice for HW7.

---
## OPTIONAL · Review examples from the lecture page

These are Examples 1–3 of the Lecture 18 page, solved with the two engines. Try each one on paper before you run its cell.

### Example 1 at a general point

For $f(x,y)=e^x\cos y$ and $\mathbf u$ at heading $\phi=\pi/4$, find $D_{\mathbf u}f(x,y)$ at a general point. (§2 and §3 did the point $(0,0)$.)

In [ ]:
d_general = directional_derivative2(sp.exp(x) * sp.cos(y), [x, y], [x, y], [a1, b1])
show_math(r'D_{\mathbf u}f(x,y) = ' + sp.latex(sp.factor(sp.expand(d_general))))

### Example 2 · Directional derivative between two points

Let $f(x,y)=x^2+2xy-y^3$.

1. Find $D_{\mathbf u}f(1,0)$ in the direction from $P(1,0)$ to $Q(0,1)$.
2. Find $\nabla f(3,2)$.

In [ ]:
f2 = x**2 + 2*x*y - y**3
P2, Q2 = (1, 0), (0, 1)
v2 = [Q2[0] - P2[0], Q2[1] - P2[1]]          # Q - P, not P - Q
show_text(f'**Part 1.** Direction vector Q − P = ⟨{v2[0]}, {v2[1]}⟩.')
D2 = directional_derivative2(f2, [x, y], list(P2), v2)
show_math(r'D_{\mathbf u}f(1,0) = ' + sp.latex(sp.simplify(D2)))

show_text('**Part 2.**')
show_math(r'\nabla f = ' + vec(compute_gradient(f2, [x, y]))
          + r',\qquad \nabla f(3,2) = ' + vec(compute_gradient(f2, [x, y], [3, 2])))

*Note.* In Part 1 the rate is $0$ while $\nabla f(1,0)=\langle2,2\rangle\neq\mathbf 0$. So the direction from $P$ to $Q$ is tangent to the level curve through $P$, exactly as in §7.

### Example 3 · Mixing gradient and directional derivatives

Let $f(x,y)=3-\dfrac{x^2}{10}+\dfrac{xy^2}{10}$.

1. Compute $\nabla f(3,-1)$.
2. Compute $D_{\mathbf u}f(3,-1)$ with $\mathbf u=\langle 1/\sqrt2,-1/\sqrt2\rangle$.
3. Compute the directional derivative at $(-3,1)$ in the direction of $\langle 3,4\rangle$.

In [ ]:
f3 = 3 - x**2/10 + x*y**2/10

show_text('**Part 1.**')
show_math(r'\nabla f(3,-1) = ' + vec(compute_gradient(f3, [x, y], [3, -1])))

show_text('**Part 2.**')
D3b = sp.simplify(directional_derivative2(f3, [x, y], [3, -1], [1/sp.sqrt(2), -1/sp.sqrt(2)]))
show_math(r'D_{\mathbf u}f(3,-1) = ' + sp.latex(D3b) + r'\approx ' + fmt(D3b, 4))

show_text('**Part 3.** The vector ⟨3, 4⟩ is not a unit vector. The engine normalizes it to ⟨3/5, 4/5⟩.')
D3c = sp.simplify(directional_derivative2(f3, [x, y], [-3, 1], [3, 4]))
show_math(r'D_{\mathbf u}f(-3,1) = ' + sp.latex(D3c) + r' = ' + fmt(D3c, 2))

---
<div style="background:#0F1A2B;border-top:2px solid #22D3EE;color:#94A3B8;padding:8px 16px;font-family:Menlo,Consolas,monospace;font-size:12px;letter-spacing:1px;">
END OF LAB 18 · ROVER FIELD TEST · © Tan H. Cao, for AMS 261 students
</div>